# 토큰화 (Tokenization)

- 문장이나 단어를 더 작은 단위로 나누어 분석 가능한 단위(토큰, Token)으로 변환하는 과정
- 토큰의 단위가 상황에 따라 다르지만, 보통 의미있는 혹은 처리하는 단위로써 토큰 정의
- 자연어 처리에서 크롤링, 데이터 수집 등으로 얻은 코퍼스 데이터는 정제되지 않은 경우가 많은데 이를 사용 용도에 맞게 토큰화, 정제, 정규화하는 과정이 필요

**토큰화 목적**

- 문법적 구조 이해
- 유연한 데이터 활용

In [2]:
%pip install transformers spacy kss konlpy tensorflow ipywidgets tqdm

     ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
     ---------------------------------------- 1.1/1.1 MB 11.0 MB/s  0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-win_amd64.whl.metadata (2.4 kB)
  Using cached anyio-4.14.2-py3-none-any.whl.metadata (4.6 kB)
  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached charset_normalizer-3.4.9-cp312-cp312-win_amd64.whl.

NLTK 기본 토큰화 -> 서브워드 토큰화 -> 규칙/정규식 기반 토크나이저 비교 -> 한국어 문장 분리 (KSS) -> 영어 품사 태깅 (NLTK, SpaCy) -> 한국어 형태소 분석 및 품사 태깅 (KoNLPy)

- NLTK 기본 토큰화
  - NLTK : 자연어 처리 라이브러리
  - NLTK 기본 토큰화 : 단어 단위 / 문장 단위로 쪼개는 작업. (띄어쓰기, 마침표, 쉼표 등 고려하여)

- 서브워드 토큰화
  - 서브워드 : 단어보다는 작고 글자보다는 큰, 단어의 조각 ex) unhappiness -> un + ha + pp + iness
  - 서브워드 토큰화 : 복잡 or 사전에 없는 낯선/새로운 단어를 더 작은 조각 단위로 쪼개서 처리하는 기법 (오타 or 신조어가 나와도 뜻 유추 가능)

- 규칙/정규식 기반 토크나이저 비교
  - 규칙/정규식 : 특정 문자 패턴 (ex) \b\w+\b -> 순수 단어 문자만)을 정의해 텍스트를 걸러내는 방식
  - 규칙/정규식 기반으로 토크나이저를 비교하는 이유 : 토크나이저마다 내부 규칙이 달라서

- 한국어 문장 분리 (KSS)
  - KSS : 한국어를 문장 단위로 쪼개는 라이브러리

- 영어 품사 태깅 (NLTK, SpaCy)
  - POS 태깅 : 영어 품사 태깅. 각 단어가 문맥상 어떤 품사로 쓰였는지 라벨을 붙여주는 작업

In [3]:
import nltk
from tqdm.auto import tqdm

nltk.download('punkt')  # 문장 / 단어 토큰화용 리소스
nltk.download('punkt_tab')  # punkt 관련 테이블 리소스

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Playdata\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Playdata\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [4]:
text = 'NLP is fascinating. It has many applications in real-world scenarios'

In [5]:
# nltk 토큰화 복습
print(nltk.word_tokenize(text))  # 단어 토큰화

print(nltk.sent_tokenize(text))  # 문장 토큰화

for sent in nltk.sent_tokenize(text):  # 문장 단위로 순회
    print(nltk.word_tokenize(sent))  # 각 문장을 단어 토큰화

['NLP', 'is', 'fascinating', '.', 'It', 'has', 'many', 'applications', 'in', 'real-world', 'scenarios']
['NLP is fascinating.', 'It has many applications in real-world scenarios']
['NLP', 'is', 'fascinating', '.']
['It', 'has', 'many', 'applications', 'in', 'real-world', 'scenarios']


### Subword Tokenization

- BertTokenizer
    - 단어를 부분 단위로 쪼개어 희귀하거나 새로운 단어도 부분적으로 표현할 수 있도록 함 -> 어휘 크기를 줄이고 다양한 언어 패턴 학습 가능

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')  # 사전 학습된 토크나이저
word = 'unhappiness'
subwords = tokenizer.tokenize(word)  # 서브워드 단위로 분해
subwords

['un', '##ha', '##pp', '##iness']

BERT 서브워드 토큰화 규칙
- 단어를 사전에 있는 조각(subword)들로 쪼개서 표현한다.
- 단어의 처음 조각은 그대로, 단어 중간/뒤에서 이어지는 조각은 ##을 붙여서 표시한다.
- 통째 단어가 없으면 가장 긴 조각부터 맞춰서 쪼개고, 끝까지 조각이 안 맞으면 [UNK](모르는 토큰)으로 처리  
  -> [OOV](처음 보는 단어)를 줄이고 희귀 단어도 조각 조합으로 다룰 수 있게 된다.


  워닝 해결 : pip install -U huggingface_hub  >  hf auth login

### 문자 단위 토큰화

In [8]:
list(word)

['u', 'n', 'h', 'a', 'p', 'p', 'i', 'n', 'e', 's', 's']

In [10]:
# 정규표현식 활용한 단어 추출
import re

re.findall(r'\b\w+\b', text)  # 단어 경계를 기준으로 word(문자/숫자/_)만 추출

['NLP',
 'is',
 'fascinating',
 'It',
 'has',
 'many',
 'applications',
 'in',
 'real',
 'world',
 'scenarios']

In [11]:
# WordPunctTokenizer와 word_tokenize 비교
from nltk.tokenize import WordPunctTokenizer, word_tokenize

text = "Don't hesitate to use well-being practices for self-care."

word_punct_tokenizer = WordPunctTokenizer()
print(word_punct_tokenizer.tokenize(text))  # 단어/구두점 기준으로 토큰화
print(word_tokenize(text))  # NLTK 기본 토큰화

['Don', "'", 't', 'hesitate', 'to', 'use', 'well', '-', 'being', 'practices', 'for', 'self', '-', 'care', '.']
['Do', "n't", 'hesitate', 'to', 'use', 'well-being', 'practices', 'for', 'self-care', '.']


WordPunctTokenizer는 ', - 같은 구두점 기준으로 더 잘게 쪼갠다   
word_tokenize는 상대적으로 조금 더 자연스러운 단어 단위로 토큰화한다

In [14]:
# TreebankTokenizer와 word_tokenize 비교
from nltk.tokenize import TreebankWordTokenizer, word_tokenize

text = "Don't hesitate to use well-being practices for self-care. NLP is fascinating. It has many applications in real-world scenarios"

treebank_tokenizer = TreebankWordTokenizer()
print(treebank_tokenizer.tokenize(text))  # 단어/구두점 기준으로 토큰화
print(word_tokenize(text))  # NLTK 기본 토큰화

['Do', "n't", 'hesitate', 'to', 'use', 'well-being', 'practices', 'for', 'self-care.', 'NLP', 'is', 'fascinating.', 'It', 'has', 'many', 'applications', 'in', 'real-world', 'scenarios']
['Do', "n't", 'hesitate', 'to', 'use', 'well-being', 'practices', 'for', 'self-care', '.', 'NLP', 'is', 'fascinating', '.', 'It', 'has', 'many', 'applications', 'in', 'real-world', 'scenarios']


토크나이저마다 각자의 규칙에 따라 토큰화를 진행한다.   
보통 차이나는 부분은 구두점/기호 처리 방식 (하이픈, 괄호, 달러표시$, 날짜 등)

### 한국어 토큰화

In [ ]:
import kss  # Korean sentences spliter 

text = '이 눈깔! 이 눈깔! 왜 나를 똑바루 바라보지 못하고 천장만 바라 보느냐! 응?! \
설렁탕을 사왔는데… 왜 먹지를 못하니? 왜 먹지를 못하니! 괴상하게도 오늘은 운수가 좋더니만…'

kss.split_sentences(text) # 문장 단위로 분리  (구두점 기준으로 분리)

[Kss]: Because there's no supported C++ morpheme analyzer, Kss will take pecab as a backend. :D
For your information, Kss also supports mecab backend.
We recommend you to install mecab or konlpy.tag.Mecab for faster execution of Kss.
Please refer to following web sites for details:
- mecab: https://cleancode-ws.tistory.com/97
- konlpy.tag.Mecab: https://uwgdqo.tistory.com/363

c:\Users\Playdata\NLP\nlp_venv\Lib\site-packages\pecab\_tokenizer.py:265: RuntimeWarning: overflow encountered in scalar add
  from_pos_data.costs[idx]


['이 눈깔!',
 '이 눈깔!',
 '왜 나를 똑바루 바라보지 못하고 천장만 바라 보느냐!',
 '응?!',
 '설렁탕을 사왔는데… 왜 먹지를 못하니?',
 '왜 먹지를 못하니!',
 '괴상하게도 오늘은 운수가 좋더니만…']

### 품사 태깅

**pos_tag**

pos_tag는 자연어 처리(NLP)에서 단어에 품사를 태깅하는 함수로, 주로 NLTK와 같은 라이브러리에서 사용된다.

- nltk pos_tag() 주요 품사 태깅<br>

1. **NN (Noun, Singular)**  
   단수 명사를 나타낸다. 하나의 사물이나 개념을 지칭한다.  
   예시: "cat", "book", "apple"

2. **NNS (Noun, Plural)**  
   복수 명사를 나타낸다. 두 개 이상의 사물이나 개념을 지칭한다.  
   예시: "cats", "books", "apples"

3. **NNP (Proper Noun, Singular)**  
   단수 고유 명사를 나타낸다. 특정한 사람, 장소 또는 조직의 이름을 지칭한다.  
   예시: "Alice", "London", "NASA"

4. **NNPS (Proper Noun, Plural)**  
   복수 고유 명사를 나타낸다. 두 개 이상의 특정한 사람, 장소 또는 조직의 이름을 지칭한다.  
   예시: "Smiths", "United Nations"

5. **VB (Verb, Base Form)**  
   동사의 원형을 나타낸다. 일반적으로 현재 시제와 함께 사용된다.  
   예시: "run", "eat", "play"

6. **VBD (Verb, Past Tense)**  
   동사의 과거형을 나타낸다.  
   예시: "ran", "ate", "played"

7. **VBG (Verb, Gerund or Present Participle)**  
   동명사 또는 현재 분사를 나타낸다. 일반적으로 "-ing" 형태이다.  
   예시: "running", "eating", "playing"

8. **VBN (Verb, Past Participle)**  
   동사의 과거 분사형을 나타낸다. 주로 완료 시제와 함께 사용된다.  
   예시: "run" (as in "has run"), "eaten", "played"

9. **VBZ (Verb, 3rd Person Singular Present)**  
   3인칭 단수 현재형 동사를 나타낸다. 주어가 3인칭 단수일 때 사용된다.  
   예시: "runs", "eats", "plays"

10. **JJ (Adjective)**  
    형용사를 나타낸다. 명사를 수식하여 그 특성을 설명한다.  
    예시: "big", "blue", "happy"

11. **JJR (Adjective, Comparative)**  
    비교급 형용사를 나타낸다. 두 개의 대상을 비교할 때 사용된다.  
    예시: "bigger", "bluer", "happier"

12. **JJS (Adjective, Superlative)**  
    최상급 형용사를 나타낸다. 세 개 이상의 대상을 비교할 때 사용된다.  
    예시: "biggest", "bluest", "happiest"

13. **RB (Adverb)**  
    부사를 나타낸다. 동사, 형용사 또는 다른 부사를 수식한다.  
    예시: "quickly", "very", "well"

14. **RBR (Adverb, Comparative)**  
    비교급 부사를 나타낸다. 두 개의 대상을 비교할 때 사용된다.  
    예시: "more quickly", "better"

15. **RBS (Adverb, Superlative)**  
    최상급 부사를 나타낸다. 세 개 이상의 대상을 비교할 때 사용된다.  
    예시: "most quickly", "best"

16. **IN (Preposition or Subordinating Conjunction)**  
    전치사 또는 종속 접속사를 나타낸다. 명사와의 관계를 나타내거나 종속절을 시작한다.  
    예시: "in", "on", "because"

17. **DT (Determiner)**  
    한정사를 나타낸다. 명사의 수와 상태를 정의한다.  
    예시: "the", "a", "some"

18. **PRP (Personal Pronoun)**  
    인칭 대명사를 나타낸다. 사람, 사물 등을 대체할 때 사용된다.  
    예시: "I", "you", "he", "they"

19. **PRP$ (Possessive Pronoun)**  
    소유 대명사를 나타낸다. 소유 관계를 나타낸다.  
    예시: "my", "your", "his", "their"

In [16]:
nltk.download('averaged_perceptron_tagger')  # nltk 기본 pos tagger 모델
nltk.download('averaged_perceptron_tagger_eng')  # 영어 pos tagger 리소스

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Playdata\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Playdata\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger_eng.zip.


True

In [18]:
from nltk.tag import pos_tag

text = 'Time flies like an arrow.'
tokens = word_tokenize(text)  # 단어 토큰화
print(tokens)
pos_tags = pos_tag(tokens)  # 각 토큰에 품사 태그 부여
pos_tags

['Time', 'flies', 'like', 'an', 'arrow', '.']


[('Time', 'NNP'),
 ('flies', 'NNS'),
 ('like', 'IN'),
 ('an', 'DT'),
 ('arrow', 'NN'),
 ('.', '.')]

**spacy 주요 품사 태깅**

| 태그 | 설명                 | 예시                 |
|------|----------------------|----------------------|
| ADJ  | 형용사               | big, nice           |
| ADP  | 전치사               | in, to, on          |
| ADV  | 부사                 | very, well          |
| AUX  | 조동사               | is, have (조동사로 사용될 때) |
| CONJ | 접속사               | and, or             |
| DET  | 한정사/관사          | the, a              |
| INTJ | 감탄사               | oh, wow             |
| NOUN | 명사                 | dog, table          |
| NUM  | 숫자                 | one, two, 3         |
| PART | 소사                 | 'to' (to fly에서), not |
| PRON | 대명사               | he, she, it         |
| PROPN| 고유명사             | John, France        |
| PUNCT| 구두점               | ., !, ?             |
| SCONJ| 종속 접속사          | because, if         |
| SYM  | 기호                 | $, %, @             |
| VERB | 동사                 | run, eat            |
| X    | 알 수 없는 품사       | 외국어 단어, 잘못된 형식 |

In [21]:
import spacy

spacy_nlp = spacy.load('en_core_web_sm')

In [24]:
tokens = spacy_nlp(text)  # spacy로 처리한 doc 생성

for token in tokens:  # 토큰 단위로 순회
    print(token.text, ':', token.pos_)  # 문자열과 품사 출력

Time : NOUN
flies : VERB
like : ADP
an : DET
arrow : NOUN
. : PUNCT


### KoNLPy

- 한국어 자연어 처리를 위한 라이브러리
- 형태소 분석, 품사 태깅, 텍스트 전처리 등 기능 지원
- 여러 형태소 분석기 중 적합한 분석기 선택 가능

In [ ]:
from konlpy.tag import Okt

text = '이 눈깔! 이 눈깔! 왜 나를 똑바루 바라보지 못하고 천장만 바라 보느냐! 응?! \
설렁탕을 사왔는데… 왜 먹지를 못하니? 왜 먹지를 못하니! 괴상하게도 오늘은 운수가 좋더니만…'

okt = Okt()  # 객체 생성 (Okt 사용시 JVM 필요)

morphs = okt.morphs(text)  # 형태소 단위로 분리
morphs

['이',
 '눈',
 '깔',
 '!',
 '이',
 '눈',
 '깔',
 '!',
 '왜',
 '나를',
 '똑바루',
 '바라보지',
 '못',
 '하고',
 '천장',
 '만',
 '바라',
 '보느냐',
 '!',
 '응',
 '?!',
 '설렁탕',
 '을',
 '사왔는데',
 '…',
 '왜',
 '먹지를',
 '못',
 '하니',
 '?',
 '왜',
 '먹지를',
 '못',
 '하니',
 '!',
 '괴상하게도',
 '오늘',
 '은',
 '운수',
 '가',
 '좋더니만',
 '…']

In [27]:
# 품사 태깅
pos_tags = okt.pos(text)  # (토큰, 품사) 형태로 반환
pos_tags

[('이', 'Noun'),
 ('눈', 'Noun'),
 ('깔', 'Verb'),
 ('!', 'Punctuation'),
 ('이', 'Noun'),
 ('눈', 'Noun'),
 ('깔', 'Verb'),
 ('!', 'Punctuation'),
 ('왜', 'Noun'),
 ('나를', 'Verb'),
 ('똑바루', 'Noun'),
 ('바라보지', 'Verb'),
 ('못', 'Noun'),
 ('하고', 'Josa'),
 ('천장', 'Noun'),
 ('만', 'Josa'),
 ('바라', 'Verb'),
 ('보느냐', 'Verb'),
 ('!', 'Punctuation'),
 ('응', 'Noun'),
 ('?!', 'Punctuation'),
 ('설렁탕', 'Noun'),
 ('을', 'Josa'),
 ('사왔는데', 'Verb'),
 ('…', 'Punctuation'),
 ('왜', 'Noun'),
 ('먹지를', 'Verb'),
 ('못', 'VerbPrefix'),
 ('하니', 'Verb'),
 ('?', 'Punctuation'),
 ('왜', 'Noun'),
 ('먹지를', 'Verb'),
 ('못', 'VerbPrefix'),
 ('하니', 'Verb'),
 ('!', 'Punctuation'),
 ('괴상하게도', 'Adjective'),
 ('오늘', 'Noun'),
 ('은', 'Josa'),
 ('운수', 'Noun'),
 ('가', 'Josa'),
 ('좋더니만', 'Adjective'),
 ('…', 'Punctuation')]

In [29]:
# 명사 추출
nouns = okt.nouns(text)
nouns

['이', '눈', '이', '눈', '왜', '똑바루', '못', '천장', '응', '설렁탕', '왜', '왜', '오늘', '운수']